# Processing cellpy batch - preparing the data


### `{{cookiecutter.project_name}}::{{cookiecutter.session_id}}`


**Experimental-id:** `{{cookiecutter.notebook_name}}`  
**Short-name:** `{{cookiecutter.session_id}}`  
**Project:** `{{cookiecutter.project_name}}`  
**By:** `{{cookiecutter.author_name}}`  
**Date:** `{{cookiecutter.date}}`


## 1. Setting up everything


Note! This template was made for `cellpy` version >= 2.1

If you are still on `cellpy` 1.x, use the `standard_1_x` cookie instead:

```
cookiecutter gh:jepegit/cellpy_cookies --directory standard_1_x
```


### Imports


In [ ]:
import cellpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cellpy.config as config
from cellpy import batch
from cellpy.utils import plotutils, helpers

%matplotlib inline
print(f"cellpy version: {cellpy.__version__}")

### Creating pages and initialise the cellpy batch object

If you need to create Journal Pages, please provide appropriate names for the project and the experiment to allow `cellpy` to build the pages.


In [ ]:
# Create the batch object
b = batch.load(
    project="{{cookiecutter.project_name}}",
    name="{{cookiecutter.session_id}}",
    batch_col="b01",  # edit this if you are not using the standard batch column
    allow_from_journal=True,  # load the journal in this folder if it exists
    force_cellpy=False,  # set to True if you would like to load only from cellpy files
    force_raw_file=False,  # set to True if you would like to load only from raw files
    force_recalc=False,  # set to True to remake step tables and summaries after loading
)

#### Alternative method

Sometimes things fail. To help in finding out why, you can do the steps one at a time
instead of letting ``batch.load`` do everything: ``Batch.from_db`` only reads the database
and builds the journal, so you can inspect ``b.pages`` before any data is loaded.

If you need finer control over *where* the data comes from, give ``batch.load`` an explicit
policy instead of the ``force_*`` flags, e.g.
``policy=batch.LoadPolicy(source=batch.SourcePreference.NEWEST)``.

In [ ]:
# # Parameters for the batch
# project = "{{cookiecutter.project_name}}"
# name = "{{cookiecutter.session_id}}"
# batch_col = "b01"  # edit this if you are not using the standard batch column

# # Read your db and build the journal pages (nothing is loaded yet)
# b = batch.Batch.from_db(name, project, batch_col=batch_col)

# # Create the apropriate folders
# b.paginate()

# # load the data (this might take some time)
# b.update()

# # save the journal in this folder
# b.save()

## 2. Examine your data


In [ ]:
# Show the journal pages
b.pages

## 3. Initial investigation of the batch experiment


### 3.1 Summaries


In [ ]:
# Plot the charge capacity and the C.E. (and resistance and rate) vs. cycle number (standard plot)
b.plot(rate=True, ir=True)

### 3.2 Remove bad cells


In [ ]:
# Get a report summarising the data (might give some clues to which cells are bad)
b.report()

#### The cells I want to remove (and why):


Make a note of the cells you remove, e.g. like this:

- cell x: missing too many data-points
- cell y: stopped after 2 cycles

Tip: remove the rows with the cells you do not want from the journal pages.
Tip for advanced users: mark the cells as bad instead.


#### Modify the pages

In `cellpy` 2 the journal pages are a read-only `polars` frame, so you no longer edit and
re-assign `b.pages`. Use `b.drop(label)` to remove a cell, or mark the bad ones with
`b.mark_as_bad(label)` and remove them all at once with `b.drop_cells_marked_bad()`.


In [ ]:
# # remove a cell you do not want (the label is the value in the filename column)
# b.drop("<cell-label>")

In [ ]:
# # or mark them as bad first, then remove all the marked cells in one go
# b.mark_as_bad("<cell-label>")
# b.drop_cells_marked_bad()

In [ ]:
# # if you have made any changes to your journal, you should save it again
# b.save()

#### Clean up summaries

You might want to add some code to edit the summaries before
processing them. You usuall dont know what you will have to edit before
you have done (at least parts of) the analysis. However, I advice that you always clean up your notebooks and put your pre-processing routines here so that it is easy to re-run the notebook (and understand it) later.

A good way of doing this is to create a yanked batch object (using `helpers.yank_outliers` or `helpers.yank_after`).

In [ ]:
# # example of dictionary used for input to yanking (the lists contains cycle-numbers that should be removed)
# bad_cycle_numbers = {
#     'cell_label_01':[4, 337, 338],
#     'cell_label_01': [4, 336, 533],
#     'cell_label_01': [4, 500],
#     'cell_label_01':[4],
#  }

In [ ]:
# # example on how to create a "yanked" batch object (note that the ``yank_outliers`` method has several default values):

# removed = helpers.yank_outliers(b,
#     zscore_limit=None,
#     low=1000.0,
#     high=6000.0,
#     filter_cols=None,
#     freeze_indexes=None,
#     remove_indexes=bad_cycle_numbers)

# # you can also save the info about removed cycles to your journal
# b.journal.session["bad_cycles"] = removed
# b.save()

#### Recalculate and link

If you have changed parameters that `cellpy` uses internally in its calculations, such as nominal capacity or mass, you will have to do a recalculation.
If you have removed cells, you will need to do a new linking.


In [ ]:
# b.recalc()
# b.link()

### 3.3 Raw data

You should always have a look at the raw-data to see if everything looks OK. I suggest using `raw_plot` to get a quick overview of the data. If you need more information you can use the `cycle_info_plot` function (remark! it is rather slow, so if you are going to plot a data-set with many cycles you should consider limiting it to render only specific cycles).


#### Quick look


In [ ]:
# label = b.cell_names[0]
# c = b.cells[label]
# plotutils.raw_plot(c)

#### Cycle information

Gives additional information (rate, steps, etc).


In [ ]:
# label = b.cell_names[0]
# c = b.cells[label]
# plotutils.cycle_info_plot(c)  # add get_axes=True if you want the figure returned

## 4. Packaging data

The notebooks are set up such that they read the journal from the current folder, and
`batch.load` already writes `cellpy_batch_{{cookiecutter.session_id}}.json` there. Run `b.save()`
again whenever you have changed the journal (dropped cells, edited the session notes).

If you would like to share the experiment with others, make it self-contained: write the
cellpy-files into `data/interim`, re-point the `cellpy_file_name` column of the journal at
them, and save the journal. Copying the files is not enough on its own - the journal would
still point at your own `cellpydatadir`, so whoever opened the notebooks would read your
originals (or fail because they do not have them). The 1.x `duplicate_cellpy_files` helper
is gone in `cellpy` 2; the recipe below replaces it.


In [ ]:
# # Save the journal in this folder again (after editing it):
# b.save()

# # Make the experiment self-contained before sharing the notebooks:
# from pathlib import Path
#
# import polars as pl
#
# destination = Path("data/interim")
# destination.mkdir(parents=True, exist_ok=True)
#
# cellpy_files = []
# for label in b.cell_names:
#     cellpy_file = destination / f"{label}.cellpy"
#     b.cells[label].save(cellpy_file, overwrite=True)
#     cellpy_files.append(str(cellpy_file))
#
# b.journal.pages = b.pages.with_columns(pl.Series("cellpy_file_name", cellpy_files))
# b.save()

## Links


### Notebooks

- notes and information [link](00_{{cookiecutter.notebook_name}}_notes.ipynb)
- processing raw data [link](01_{{cookiecutter.notebook_name}}_loader.ipynb)
- life [link](02_{{cookiecutter.notebook_name}}_life.ipynb)
- cycles [link](03_{{cookiecutter.notebook_name}}_cycles.ipynb)
- ica [link](04_{{cookiecutter.notebook_name}}_ica.ipynb)
- plots [link](05_{{cookiecutter.notebook_name}}_plots.ipynb)
